In [1]:
### ARTS as the line-by-line code
import os
# set global variables BEFORE importing ARTS
os.environ['ARTS_DATA_PATH'] = '../arts-cat-data-2.6.14:../arts-xml-data-2.6.14'

import pyarts
from pyarts.arts import convert
# the fluxsim module
import FluxSimulator as fsm
import seaborn as sns


# helper packages
import numpy as np
import xarray as xr
from scipy.constants import speed_of_light as c
import matplotlib.pyplot as plt

import typhon
import typhon.constants as tpc
import pandas as pd

from scipy.optimize import minimize

from cycler import cycler

In [2]:
xr.open_dataset('/home/pczarnec/.cache/pyrte_rrtmgp/rrtmgp-data-1.9/rrtmgp-gas-sw-g112.nc').compute()


<xarray.Dataset> Size: 8MB
Dimensions:                          (minor_absorber_intervals_upper: 24,
                                      minor_absorber_intervals_lower: 34,
                                      bnd: 14, atmos_layer: 2, pair: 2,
                                      temperature: 14, pressure_interp: 60,
                                      mixing_fraction: 9, gpt: 112,
                                      minor_absorber: 21, absorber: 19,
                                      pressure: 59, absorber_ext: 20,
                                      contributors_lower: 297,
                                      contributors_upper: 215)
Coordinates: (12/13)
  * minor_absorber_intervals_upper   (minor_absorber_intervals_upper) int32 96B ...
  * minor_absorber_intervals_lower   (minor_absorber_intervals_lower) int32 136B ...
  * bnd                              (bnd) int32 56B 0 1 2 3 4 ... 9 10 11 12 13
  * atmos_layer                      (atmos_layer) int32 8B 0 1
  * pair                             (pair) int32 8B 0 1
  * temperature                      (temperature) int32 56B 0 1 2 ... 11 12 13
    ...                               ...
  * mixing_fraction                  (mixing_fraction) int32 36B 0 1 2 ... 6 7 8
  * gpt                              (gpt) int32 448B 0 1 2 3 ... 109 110 111
  * pressure                         (pressure) int32 236B 0 1 2 3 ... 56 57 58
  * absorber_ext                     (absorber_ext) int32 80B 0 1 2 ... 17 18 19
  * contributors_lower               (contributors_lower) int32 1kB 0 1 ... 296
  * contributors_upper               (contributors_upper) int32 860B 0 1 ... 214
Dimensions without coordinates: minor_absorber, absorber
Data variables: (12/35)
    key_species                      (bnd, atmos_layer, pair) int32 224B 1 ... 7
    kmajor                           (temperature, pressure_interp, mixing_fraction, gpt) float64 7MB ...
    rayl_lower                       (temperature, mixing_fraction, gpt) float64 113kB ...
    rayl_upper                       (temperature, mixing_fraction, gpt) float64 113kB ...
    absorption_coefficient_ref_P     float64 8B 1.013e+05
    absorption_coefficient_ref_T     float64 8B 296.0
    ...                               ...
    solar_source_quiet               (gpt) float64 896B 6.125 1.934 ... 1.486
    kminor_lower                     (temperature, mixing_fraction, contributors_lower) float64 299kB ...
    kminor_upper                     (temperature, mixing_fraction, contributors_upper) float64 217kB ...
    tsi_default                      float64 8B 1.361e+03
    mg_default                       float32 4B 0.1568
    sb_default                       float32 4B 902.7

In [3]:
ddq_sw = xr.open_dataset('/homedata/pczarnec/ddq-implementation-sprint/DDQ configurations/DDQ_SW_source.h5', engine = "netcdf4").compute()
ddq_lw = xr.open_dataset('/homedata/pczarnec/ddq-implementation-sprint/DDQ configurations/DDQ_LW.h5', engine = "netcdf4").compute()

getfattr: /homedata/pczarnec/ddq-implementation-sprint/DDQ: No such file or directory
getfattr: configurations/DDQ_SW_source.h5: No such file or directory
getfattr: /homedata/pczarnec/ddq-implementation-sprint/DDQ: No such file or directory
getfattr: configurations/DDQ_LW.h5: No such file or directory


In [17]:
f_grid = convert.kaycm2freq(ddq_sw.S.data)

ws = pyarts.Workspace()
ws.f_grid = f_grid
ws.stokes_dim = 1

ws.rtp_pressure = 101325.0     # Pa  (1 atm)
ws.rtp_temperature = 273.15    # K   (0 C)

ws.gas_scattering_coefAirSimple()

data = np.asarray(ws.gas_scattering_coef.value.data)
beta_sca = data[0, 0, :, 0]  # [1/m], one value per frequency in f_grid

k_B = 1.380649e-23  # J/K
n_number_density = ws.rtp_pressure.value / (k_B * ws.rtp_temperature.value)  # [1/m^3]
 
sigma_sca = beta_sca / n_number_density  # [m^2] per molecule

In [20]:
ddq_sw = ddq_sw.assign({"Rayleigh_xsec" : (("S"), sigma_sca)})
ddq_sw

<xarray.Dataset> Size: 2kB
Dimensions:        (S: 64)
Coordinates:
  * S              (S) float64 512B 2.144e+03 2.411e+03 ... 4.426e+04 4.839e+04
Data variables:
    W              (S) float64 512B 1.509e+03 1.244e+03 ... 0.01592 0.01849
    solar_source   (S) float64 512B 456.1 596.5 838.6 ... 9.108 9.925 2.085
    Rayleigh_xsec  (S) float64 512B 8.392e-35 1.343e-34 ... 2.008e-29 3.076e-29
Attributes:
    description:  64 optimized wavenumbers and weights, in the shortwave, opt...

In [23]:
ddq_sw.to_netcdf("../DDQ configurations/DDQ_SW_source_Rayleigh.h5")